# 15 — Gold set, projection accuracy, and model accuracy

Assembles the adjudicated gold standard, then produces the two numbers this
exercise was for:

1. **Projection accuracy** — how often the silver labels agree with human
   judgment. Every recognition figure in the paper has so far measured
   agreement with the projection; this says what that was worth.
2. **Model accuracy against gold** — the three encoders scored against human
   annotation rather than against their own training distribution.

The gold sentences come from the NER test split, so no model saw them in
training.

**Run from the repository root.** Kernel: `Python (tka)`. A few minutes.

## Cell 1: Load the adjudicated file

In [1]:
from pathlib import Path
import json, sys
from collections import Counter, defaultdict
import numpy as np
from openpyxl import load_workbook

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ANN  = ROOT / "annotation"
RES  = ROOT / "results" / "ner"
MODELS = ROOT / "models"

adj = ANN / "adjudication.xlsx"
key = json.load(open(ANN / "gold_sample_key.json", encoding="utf-8"))
print(("  ok   " if adj.exists() else "  MISS ") + adj.name)
if not adj.exists():
    sys.exit("Run notebook 14 and complete the adjudication first")

TYPES = ["PERSON","GPE","ORG","NORP","LOC","LANGUAGE",
         "WORK_OF_ART","FAC","PRODUCT","EVENT","LAW"]
VALID = {"O"} | {f"{p}-{e}" for e in TYPES for p in ("B","I")}

ws = load_workbook(adj, data_only=True)["Adjudicate"]
final = {}
blank = bad = 0
for r in ws.iter_rows(min_row=2, values_only=True):
    if r[0] is None:
        continue
    sent, tok, word, tag = int(r[0]), int(r[2]), r[3], (r[6] or "").strip()
    if not tag:
        blank += 1; continue
    if tag not in VALID:
        bad += 1; continue
    final[(sent, tok)] = tag

print(f"adjudicated decisions : {len(final):,}")
print(f"blank FINAL cells     : {blank}")
print(f"invalid FINAL tags    : {bad}")
if blank or bad:
    print("\nFix these in adjudication.xlsx before continuing.")

  ok   adjudication.xlsx
adjudicated decisions : 1,602
blank FINAL cells     : 0
invalid FINAL tags    : 0


## Cell 2: Assemble the gold set

Sentences that had no disagreement do not appear in the adjudication file; for
those, both annotators agreed and their shared tag is the gold tag. The
adjudicated decisions override wherever they exist.

In [2]:
books = [b for b in sorted(ANN.glob("mizo_ner_annotation_*.xlsx"))
         if not b.name.startswith("~$")]
names = [b.stem.replace("mizo_ner_annotation_", "") for b in books]

def read_book(path):
    w = load_workbook(path, data_only=True)["Annotate"]
    out = {}
    for r in w.iter_rows(min_row=2, values_only=True):
        if r[0] is None: continue
        out[(int(r[0]), int(r[2]))] = (r[4] or "O").strip() or "O"
    return out

ann = {n: read_book(b) for n, b in zip(names, books)}
a, b_ = names[0], names[1]

def fix_bio(tags):
    out, prev = [], "O"
    for t in tags:
        if t.startswith("I-"):
            typ = t[2:]
            if prev not in (f"B-{typ}", f"I-{typ}"):
                t = f"B-{typ}"
        out.append(t); prev = t
    return out

gold, source = [], Counter()
for k in key:
    row = []
    for j in range(len(k["tokens"])):
        pos = (k["key_id"], j + 1)
        if pos in final:
            row.append(final[pos]); source["adjudicated"] += 1
        else:
            ta = ann[a].get(pos, "O"); tb = ann[b_].get(pos, "O")
            if ta == tb:
                row.append(ta); source["agreed"] += 1
            else:
                row.append(ta); source["unresolved (took %s)" % a] += 1
    gold.append(fix_bio(row))

print("gold tag provenance:")
for k_, v in source.most_common():
    print(f"  {k_:<26}{v:>7,}")

n_ent = sum(1 for s in gold for t in s if t.startswith("B-"))
print(f"\ngold: {len(gold)} sentences, {sum(len(s) for s in gold):,} tokens, "
      f"{n_ent:,} entities")
print(f"\n{'Type':<14}{'entities':>9}")
for t, c in Counter(x[2:] for s in gold for x in s if x.startswith("B-")).most_common():
    print(f"{t:<14}{c:>9,}")

out_gold = [{"key_id": k["key_id"], "corpus_id": k["corpus_id"],
             "tokens": k["tokens"], "tags": g} for k, g in zip(key, gold)]
with open(ANN / "mizo_ner_gold.json", "w", encoding="utf-8") as f:
    json.dump(out_gold, f, ensure_ascii=False, indent=1)
print(f"\n-> annotation/mizo_ner_gold.json")

gold tag provenance:
  agreed                      2,042
  adjudicated                 1,602

gold: 300 sentences, 3,644 tokens, 420 entities

Type           entities
PERSON              237
GPE                  66
ORG                  53
LOC                  40
NORP                 17
LANGUAGE              4
EVENT                 2
FAC                   1

-> annotation/mizo_ner_gold.json


## Cell 3: Projection accuracy

The headline result. Silver labels are scored against human judgment on the
same sentences. Because the projection marks stems while annotators tagged
whole tokens, spans are compared by overlap: a projected span counts as correct
if it overlaps a gold span of the same type.

In [3]:
from seqeval.metrics import (classification_report, f1_score,
                             precision_score, recall_score)

def spans(tags):
    out, i = [], 0
    while i < len(tags):
        if tags[i].startswith("B-"):
            lab = tags[i][2:]; j = i + 1
            while j < len(tags) and tags[j] == f"I-{lab}": j += 1
            out.append((i, j, lab)); i = j
        else:
            i += 1
    return out

def overlap_prf(ref_seqs, hyp_seqs):
    tp = fp = fn = 0
    per = defaultdict(lambda: [0, 0, 0])
    for rt, ht in zip(ref_seqs, hyp_seqs):
        R, H = spans(rt), spans(ht)
        usedR = set()
        for (hs, he, hl) in H:
            hit = None
            for idx, (rs, re_, rl) in enumerate(R):
                if idx in usedR or rl != hl:
                    continue
                if rs < he and re_ > hs:
                    hit = idx; break
            if hit is None:
                fp += 1; per[hl][1] += 1
            else:
                usedR.add(hit); tp += 1; per[hl][0] += 1
        for idx, (rs, re_, rl) in enumerate(R):
            if idx not in usedR:
                fn += 1; per[rl][2] += 1
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f, tp, fp, fn, per

proj = [k["projection_tags"] for k in key]
p, r, f, tp, fp, fn, per = overlap_prf(gold, proj)

print("PROJECTION vs HUMAN GOLD (overlap matching)")
print(f"  precision {p:.4f}   recall {r:.4f}   F1 {f:.4f}")
print(f"  matched {tp:,}   spurious {fp:,}   missed {fn:,}\n")
print(f"{'Type':<14}{'gold':>7}{'proj':>7}{'P':>9}{'R':>9}{'F1':>9}")
print("-" * 55)
for t in sorted(per, key=lambda x: -(per[x][0] + per[x][2])):
    tpc, fpc, fnc = per[t]
    pp = tpc/(tpc+fpc) if tpc+fpc else 0
    rr = tpc/(tpc+fnc) if tpc+fnc else 0
    ff = 2*pp*rr/(pp+rr) if pp+rr else 0
    print(f"{t:<14}{tpc+fnc:>7}{tpc+fpc:>7}{pp:>9.4f}{rr:>9.4f}{ff:>9.4f}")

print(f"""
The paper reports micro-F1 0.8739 for XLM-RoBERTa against silver labels.
Projection accuracy of {f:.4f} is the ceiling that figure was measured against.
""")

PROJECTION vs HUMAN GOLD (overlap matching)
  precision 0.6300   recall 0.6000   F1 0.6146
  matched 252   spurious 148   missed 168

Type             gold   proj        P        R       F1
-------------------------------------------------------
PERSON            237    224   0.8348   0.7890   0.8113
GPE                66     75   0.4800   0.5455   0.5106
ORG                53     64   0.3750   0.4528   0.4103
LOC                40      5   0.2000   0.0250   0.0444
NORP               17     17   0.1176   0.1176   0.1176
LANGUAGE            4      2   1.0000   0.5000   0.6667
EVENT               2      1   0.0000   0.0000   0.0000
FAC                 1      5   0.0000   0.0000   0.0000
WORK_OF_ART         0      5   0.0000   0.0000   0.0000
PRODUCT             0      2   0.0000   0.0000   0.0000

The paper reports micro-F1 0.8739 for XLM-RoBERTa against silver labels.
Projection accuracy of 0.6146 is the ceiling that figure was measured against.



## Cell 4: Model accuracy against gold

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
CAND = {"xlmr": (MODELS/"mizo_ner_v2", 96),
        "mizbert": (MODELS/"mizo_ner_mizbert", 64),
        "mbert": (MODELS/"mizo_ner_mbert", 96)}
AVAIL = {k: v for k, v in CAND.items() if (v[0]/"config.json").exists()}
print("models:", list(AVAIL) or "none found")

toks_all = [k["tokens"] for k in key]

def predict(d, max_len, batch=64):
    tok = AutoTokenizer.from_pretrained(str(d))
    m = AutoModelForTokenClassification.from_pretrained(str(d)).to(device).eval()
    id2 = {int(x): y for x, y in m.config.id2label.items()}
    out = []
    for i in range(0, len(toks_all), batch):
        ch = toks_all[i:i+batch]
        enc = tok(ch, is_split_into_words=True, max_length=max_len,
                  padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            pr = torch.argmax(m(**enc).logits, dim=2).cpu().numpy()
        for j, t in enumerate(ch):
            wid, seen, row = enc.word_ids(batch_index=j), set(), ["O"]*len(t)
            for pos, w in enumerate(wid):
                if w is not None and w not in seen:
                    seen.add(w); row[w] = id2[int(pr[j][pos])]
            out.append(row)
    del m; torch.cuda.empty_cache()
    return out

SILVER = {"xlmr": 0.8739, "mizbert": 0.8788, "mbert": 0.8810}
preds, model_res = {}, {}
print(f"\n{'Model':<10}{'P':>9}{'R':>9}{'F1 gold':>10}{'F1 silver':>11}{'gap':>9}")
print("-" * 58)
for k_, (d, ml) in AVAIL.items():
    preds[k_] = predict(d, ml)
    pp, rr, ff, *_ = overlap_prf(gold, preds[k_])
    s = SILVER.get(k_)
    model_res[k_] = {"precision": round(pp,4), "recall": round(rr,4),
                     "f1_gold": round(ff,4), "f1_silver": s}
    print(f"{k_:<10}{pp:>9.4f}{rr:>9.4f}{ff:>10.4f}{s:>11.4f}{ff-s:>+9.4f}")

models: ['xlmr', 'mizbert', 'mbert']

Model             P        R   F1 gold  F1 silver      gap
----------------------------------------------------------
xlmr         0.6263   0.5905    0.6078     0.8739  -0.2661
mizbert      0.6599   0.6238    0.6414     0.8788  -0.2374
mbert        0.6200   0.5905    0.6049     0.8810  -0.2761


## Cell 5: Does the model beat its own training signal?

The projection is the model's only teacher. If the model scores higher against
gold than the projection does, it has generalised past the noise it was trained
on rather than merely reproducing it.

In [5]:
print(f"{'Source of labels':<26}{'F1 vs gold':>12}")
print("-" * 40)
print(f"{'projection (silver)':<26}{f:>12.4f}")
for k_ in preds:
    print(f"{k_ + ' (trained on silver)':<26}{model_res[k_]['f1_gold']:>12.4f}")

best = max(model_res, key=lambda x: model_res[x]["f1_gold"]) if model_res else None
if best:
    d = model_res[best]["f1_gold"] - f
    print()
    if d > 0.01:
        print(f"{best} exceeds the projection by {d:+.4f}. The model has learned")
        print("regularities the projection applies inconsistently, so training on")
        print("noisy labels produced something better than the labels.")
    elif d < -0.01:
        print(f"{best} falls short of the projection by {d:+.4f}: the model has not")
        print("fully absorbed even the silver signal.")
    else:
        print(f"{best} and the projection are within 0.01 - the model reproduces")
        print("its training signal closely, neither improving on it nor losing it.")

Source of labels            F1 vs gold
----------------------------------------
projection (silver)             0.6146
xlmr (trained on silver)        0.6078
mizbert (trained on silver)      0.6414
mbert (trained on silver)       0.6049

mizbert exceeds the projection by +0.0268. The model has learned
regularities the projection applies inconsistently, so training on
noisy labels produced something better than the labels.


## Cell 6: Save and emit LaTeX

In [6]:
out = {
    "gold_sentences": len(gold),
    "gold_tokens": int(sum(len(s) for s in gold)),
    "gold_entities": int(n_ent),
    "matching": "overlap (projection marks stems, annotators tagged whole tokens)",
    "projection_vs_gold": {"precision": round(p,4), "recall": round(r,4),
                           "f1": round(f,4), "matched": tp,
                           "spurious": fp, "missed": fn},
    "projection_per_type": {t: {"gold": per[t][0]+per[t][2],
                                "predicted": per[t][0]+per[t][1],
                                "tp": per[t][0], "fp": per[t][1], "fn": per[t][2]}
                            for t in per},
    "models_vs_gold": model_res,
}
with open(RES / "gold_standard_evaluation.json", "w", encoding="utf-8") as fh:
    json.dump(out, fh, ensure_ascii=False, indent=2)
print(f"-> results/ner/gold_standard_evaluation.json\n")

print("% ---- Table: accuracy against human annotation ----")
print(f"Projection (silver labels) & {p:.4f} & {r:.4f} & {f:.4f} & --- \\\\")
NM = {"xlmr":"XLM-RoBERTa base","mizbert":"MizBERT","mbert":"mBERT cased"}
for k_ in preds:
    m = model_res[k_]
    print(f"{NM.get(k_,k_)} & {m['precision']:.4f} & {m['recall']:.4f} "
          f"& {m['f1_gold']:.4f} & {m['f1_silver']:.4f} \\\\")

-> results/ner/gold_standard_evaluation.json

% ---- Table: accuracy against human annotation ----
Projection (silver labels) & 0.6300 & 0.6000 & 0.6146 & --- \\
XLM-RoBERTa base & 0.6263 & 0.5905 & 0.6078 & 0.8739 \\
MizBERT & 0.6599 & 0.6238 & 0.6414 & 0.8788 \\
mBERT cased & 0.6200 & 0.5905 & 0.6049 & 0.8810 \\


## Cell 7

In [7]:
merged_gold = [[t.replace("-GPE","-LOCATION").replace("-LOC","-LOCATION") for t in s]
               for s in gold]
merged_proj = [[t.replace("-GPE","-LOCATION").replace("-LOC","-LOCATION") for t in s]
               for s in proj]
p2, r2, f2, tp2, fp2, fn2, per2 = overlap_prf(merged_gold, merged_proj)
print(f"projection, GPE and LOC separate : F1 {f:.4f}")
print(f"projection, merged to LOCATION   : F1 {f2:.4f}   (+{f2-f:.4f})")

for k_ in preds:
    mp = [[t.replace("-GPE","-LOCATION").replace("-LOC","-LOCATION") for t in s]
          for s in preds[k_]]
    _,_,ff,*_ = overlap_prf(merged_gold, mp)
    print(f"  {k_:<10} separate {model_res[k_]['f1_gold']:.4f}  merged {ff:.4f}")

# and: how much of the remainder is detection vs typing?
def decompose(ref, hyp):
    both = typed = 0
    for rt, ht in zip(ref, hyp):
        R = {(s,e):l for s,e,l in spans(rt)}
        H = {(s,e):l for s,e,l in spans(ht)}
        for (hs,he), hl in H.items():
            for (rs,re_), rl in R.items():
                if rs < he and re_ > hs:
                    both += 1
                    if rl != hl: typed += 1
                    break
    print(f"\noverlapping spans: {both}, of which wrong type: {typed} "
          f"({typed/both*100:.0f}%)")
decompose(gold, proj)

projection, GPE and LOC separate : F1 0.6146
projection, merged to LOCATION   : F1 0.6146   (+0.0000)
  xlmr       separate 0.6078  merged 0.6078
  mizbert    separate 0.6414  merged 0.6414
  mbert      separate 0.6049  merged 0.6049

overlapping spans: 368, of which wrong type: 115 (31%)


# Cell 8

In [8]:
def merge_place(t):
    if t == "O" or "-" not in t:
        return t
    pre, typ = t.split("-", 1)
    return f"{pre}-LOCATION" if typ in ("GPE", "LOC") else t

mg = [[merge_place(t) for t in s] for s in gold]
mp = [[merge_place(t) for t in s] for s in proj]

# sanity: the mapping must actually collapse the two
before = {t for s in gold+proj for t in s if t.endswith(("GPE","LOC"))}
after  = {t for s in mg+mp for t in s if "LOCATION" in t}
print("before:", sorted(before))
print("after :", sorted(after))

p2, r2, f2, tp2, fp2, fn2, _ = overlap_prf(mg, mp)
print(f"\nprojection, GPE/LOC separate : F1 {f:.4f}")
print(f"projection, merged           : F1 {f2:.4f}   ({f2-f:+.4f})")

for k_ in preds:
    mm = [[merge_place(t) for t in s] for s in preds[k_]]
    _, _, ff, *_ = overlap_prf(mg, mm)
    print(f"  {k_:<10} separate {model_res[k_]['f1_gold']:.4f}   merged {ff:.4f}"
          f"   ({ff-model_res[k_]['f1_gold']:+.4f})")

before: ['B-GPE', 'B-LOC', 'I-GPE', 'I-LOC']
after : ['B-LOCATION', 'I-LOCATION']

projection, GPE/LOC separate : F1 0.6146
projection, merged           : F1 0.6561   (+0.0415)
  xlmr       separate 0.6078   merged 0.6593   (+0.0515)
  mizbert    separate 0.6414   merged 0.6928   (+0.0514)
  mbert      separate 0.6049   merged 0.6537   (+0.0488)


# Cell 9

In [16]:
from pathlib import Path
import json
from openpyxl import load_workbook
from collections import Counter

ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
ANN = ROOT / "annotation"

p = ANN / "adjudication.xlsx"
import datetime
print("adjudication.xlsx modified:",
      datetime.datetime.fromtimestamp(p.stat().st_mtime))
print("Excel lock file present:", (ANN / "~$adjudication.xlsx").exists())

ws = load_workbook(p, data_only=True)["Adjudicate"]
diff = same = 0
finals = Counter()
for r in ws.iter_rows(min_row=2, values_only=True):
    if r[0] is None: continue
    ta, tb, fin = r[4], r[5], (r[6] or "").strip()
    finals[fin] += 1
    if ta != tb:
        if fin == ta: same += 1
        else: diff += 1
print(f"\ndisagreement rows where FINAL != annotator A : {diff}")
print(f"disagreement rows where FINAL == annotator A : {same}")
print("\nFINAL column value counts:", dict(finals.most_common(8)))

adjudication.xlsx modified: 2026-08-28 07:39:58.664040
Excel lock file present: False

disagreement rows where FINAL != annotator A : 77
disagreement rows where FINAL == annotator A : 102

FINAL column value counts: {'O': 1342, 'B-PERSON': 59, 'B-GPE': 57, 'B-ORG': 34, 'I-ORG': 25, 'B-LOC': 22, 'I-PERSON': 21, 'B-NORP': 17}
